# 📈 Advanced AI Stock & ETF Analyzer — Gradio UI
**Powered by Claude AI · Yahoo Finance · Real-time Data**

Works in **Google Colab** and **VS Code** (with Jupyter extension).

| Environment | API Key Setup |
|---|---|
| **Google Colab** | Add `ANTHROPIC_API_KEY` to Colab Secrets (🔑 icon in sidebar) |
| **VS Code / Local** | Set env var `ANTHROPIC_API_KEY=sk-ant-...` or create a `.env` file in the project folder |

Run cells top to bottom. The Gradio app launches in the last cell.

In [ ]:
# ── Environment Detection ─────────────────────────────────────
# Detects Google Colab vs VS Code / local Jupyter automatically
try:
    from google.colab import userdata  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ── Install Dependencies ──────────────────────────────────────
# Works in both Colab and VS Code Jupyter
import subprocess, sys
subprocess.run(
    [sys.executable, "-m", "pip", "install",
     "anthropic", "yfinance", "pandas", "matplotlib",
     "requests", "beautifulsoup4", "gradio", "-q"],
    capture_output=True
)
print("✅ Packages installed")

# ── Core Imports ──────────────────────────────────────────────
import anthropic
import yfinance as yf
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # Non-interactive backend — works in both Colab and VS Code
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import requests
import gradio as gr
from bs4 import BeautifulSoup
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings("ignore")

# ── Load API Key ──────────────────────────────────────────────
if IN_COLAB:
    # Colab: use Secrets (🔑 icon in left sidebar → add ANTHROPIC_API_KEY)
    ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
else:
    # VS Code / Local: try environment variable, then .env file
    import os
    from dotenv import load_dotenv
    load_dotenv()
    ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "")

In [2]:

# ════════════════════════════════════════════════════════════
# 😨😎 FEAR & GREED INDEX
# ════════════════════════════════════════════════════════════

def get_fear_greed():
    """
    Fetch the Fear & Greed Index using multiple fallback sources:
    1. fear-and-greed PyPI package (scrapes CNN directly)
    2. CNN dataviz API with extended headers
    3. VIX-based market proxy (calculated from live data)
    """

    def score_to_meta(score):
        score = round(score)
        if score <= 25:
            return "Extreme Fear", "😱", "Extreme Fear — market deeply oversold; historically a contrarian buying opportunity"
        elif score <= 45:
            return "Fear", "😨", "Fear — investors cautious; often precedes market recoveries"
        elif score <= 55:
            return "Neutral", "😐", "Neutral — balanced sentiment, no strong directional bias"
        elif score <= 75:
            return "Greed", "😊", "Greed — bullish momentum building; watch for overextension"
        else:
            return "Extreme Greed", "🤑", "Extreme Greed — market frothy; elevated pullback risk"

    # ── Source 1: fear-and-greed PyPI package ────────────────
    try:
        import subprocess, sys
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "fear-and-greed", "-q"],
            capture_output=True
        )
        import fear_and_greed
        data    = fear_and_greed.get()
        score   = round(float(data.value))
        label, emoji, mood = score_to_meta(score)
        return {
            "score": score, "label": label, "emoji": emoji, "mood": mood,
            "prev_week": None, "prev_month": None,
            "trend": "N/A", "source": "CNN via fear-and-greed package",
            "error": None
        }
    except Exception:
        pass

    # ── Source 2: CNN dataviz API — multiple header combos ───
    endpoints = [
        "https://production.dataviz.cnn.io/index/fearandgreed/graphdata",
        "https://production.dataviz.cnn.io/index/fearandgreed/graphdata/current",
    ]
    header_sets = [
        {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
            "Accept": "application/json, text/plain, */*",
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://www.cnn.com/markets/fear-and-greed",
            "Origin": "https://www.cnn.com",
        },
        {
            "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 (KHTML, like Gecko) Version/17.0 Safari/605.1.15",
            "Accept": "application/json",
        },
    ]
    for url in endpoints:
        for headers in header_sets:
            try:
                r    = requests.get(url, headers=headers, timeout=10)
                if r.status_code == 200:
                    data    = r.json()
                    current = data.get("fear_and_greed", {})
                    score   = round(float(current.get("score", 0)))
                    if score == 0:
                        continue
                    label, emoji, mood = score_to_meta(score)

                    hist        = data.get("fear_and_greed_historical", {}).get("data", [])
                    prev_week   = round(float(hist[-8]["x"]))  if len(hist) >= 8  else None
                    prev_month  = round(float(hist[-31]["x"])) if len(hist) >= 31 else None
                    trend       = f"{'▲' if prev_week and score > prev_week else '▼'} {abs(score - prev_week)} pts vs last week" if prev_week else "N/A"

                    return {
                        "score": score, "label": label, "emoji": emoji, "mood": mood,
                        "prev_week": prev_week, "prev_month": prev_month,
                        "trend": trend, "source": "CNN dataviz API",
                        "error": None
                    }
            except Exception:
                continue

    # ── Source 3: VIX-based proxy (always available) ─────────
    try:
        vix_df = yf.Ticker("^VIX").history(period="35d")
        spy_df = yf.Ticker("SPY").history(period="35d")

        if not vix_df.empty and not spy_df.empty:
            vix_now  = vix_df["Close"].iloc[-1]
            vix_week = vix_df["Close"].iloc[-6]  if len(vix_df) >= 6  else vix_now
            vix_month= vix_df["Close"].iloc[-22] if len(vix_df) >= 22 else vix_now

            spy_ret_1w = (spy_df["Close"].iloc[-1] / spy_df["Close"].iloc[-6]  - 1) * 100 if len(spy_df) >= 6  else 0
            spy_ret_1m = (spy_df["Close"].iloc[-1] / spy_df["Close"].iloc[-22] - 1) * 100 if len(spy_df) >= 22 else 0

            # Inverse VIX score (VIX ~10=100, VIX ~45=0)
            vix_score      = max(0, min(100, round(100 - ((vix_now - 10) / 35) * 100)))
            momentum_bonus = max(-20, min(20, round(spy_ret_1m * 2)))
            proxy_score    = max(0, min(100, vix_score + momentum_bonus))

            label, emoji, mood = score_to_meta(proxy_score)

            # Week-ago and month-ago proxy scores
            vix_score_week  = max(0, min(100, round(100 - ((vix_week  - 10) / 35) * 100)))
            vix_score_month = max(0, min(100, round(100 - ((vix_month - 10) / 35) * 100)))

            delta = proxy_score - vix_score_week
            trend = f"{'▲' if delta >= 0 else '▼'} {abs(delta)} pts vs last week"

            return {
                "score":      proxy_score,
                "label":      label,
                "emoji":      emoji,
                "mood":       mood + f" (VIX: {vix_now:.1f})",
                "prev_week":  vix_score_week,
                "prev_month": vix_score_month,
                "trend":      trend,
                "source":     "⚠️ VIX-based proxy (CNN unavailable)",
                "error":      None
            }
    except Exception:
        pass

    # ── All sources failed ────────────────────────────────────
    return {
        "score": None, "label": "Unavailable", "emoji": "❓",
        "mood": "Could not fetch Fear & Greed data from any source.",
        "prev_week": None, "prev_month": None,
        "trend": "N/A", "source": "None", "error": "All sources failed"
    }

def get_stock_data(ticker):
    try:
        stock = yf.Ticker(ticker)
        df = stock.history(period="2y")
        if df.empty:
            return None, None, "Could not fetch data. Check ticker symbol."
        return df, stock.info, None
    except Exception as e:
        return None, None, str(e)

def is_etf(info):
    return info.get("quoteType", "").upper() == "ETF"

def get_fundamentals(info):
    def safe(val):
        try:
            return f"{val:.2f}" if val else "N/A"
        except:
            return "N/A"

    # ── Analyst Rating — use recommendationMean (numeric) as primary ──
    # yfinance recommendationKey is unreliable and often defaults to "strong_buy"
    # recommendationMean: 1.0=Strong Buy, 2.0=Buy, 3.0=Hold, 4.0=Sell, 5.0=Strong Sell
    def parse_analyst_rating(info):
        mean  = info.get("recommendationMean")
        key   = info.get("recommendationKey", "")
        count = info.get("numberOfAnalystOpinions")

        if mean is not None:
            try:
                mean = float(mean)
                if   mean <= 1.5: label = "Strong Buy"
                elif mean <= 2.5: label = "Buy"
                elif mean <= 3.5: label = "Hold"
                elif mean <= 4.5: label = "Sell"
                else:             label = "Strong Sell"
                count_str = f" ({count} analysts)" if count else ""
                return f"{label}{count_str}  [score: {mean:.1f}/5.0]"
            except Exception:
                pass

        # Fallback to recommendationKey only if mean is missing
        if key and key.lower() not in ("", "none", "n/a"):
            readable = key.replace("_", " ").title()
            count_str = f" ({count} analysts)" if count else ""
            return f"{readable}{count_str}"

        return "N/A"

    return {
        "Market Cap":      f"${info.get('marketCap',0)/1e9:.2f}B" if info.get('marketCap') else "N/A",
        "P/E Ratio":       safe(info.get("trailingPE")),
        "Forward P/E":     safe(info.get("forwardPE")),
        "PEG Ratio":       safe(info.get("pegRatio")),
        "Price/Book":      safe(info.get("priceToBook")),
        "EV/EBITDA":       safe(info.get("enterpriseToEbitda")),
        "Revenue (TTM)":   f"${info.get('totalRevenue',0)/1e9:.2f}B" if info.get('totalRevenue') else "N/A",
        "Revenue Growth":  f"{info.get('revenueGrowth',0)*100:.1f}%" if info.get('revenueGrowth') else "N/A",
        "Gross Margin":    f"{info.get('grossMargins',0)*100:.1f}%" if info.get('grossMargins') else "N/A",
        "Net Margin":      f"{info.get('profitMargins',0)*100:.1f}%" if info.get('profitMargins') else "N/A",
        "ROE":             f"{info.get('returnOnEquity',0)*100:.1f}%" if info.get('returnOnEquity') else "N/A",
        "Debt/Equity":     safe(info.get("debtToEquity")),
        "Free Cash Flow":  f"${info.get('freeCashflow',0)/1e9:.2f}B" if info.get('freeCashflow') else "N/A",
        "Dividend Yield":  f"{info.get('dividendYield',0)*100:.2f}%" if info.get('dividendYield') else "N/A",
        "52W High":        safe(info.get("fiftyTwoWeekHigh")),
        "52W Low":         safe(info.get("fiftyTwoWeekLow")),
        "Analyst Target":  safe(info.get("targetMeanPrice")),
        "Target High":     safe(info.get("targetHighPrice")),
        "Target Low":      safe(info.get("targetLowPrice")),
        "Analyst Rating":  parse_analyst_rating(info),
    }

def get_etf_info(info):
    return {
        "Fund Family":   info.get("fundFamily","N/A"),
        "Category":      info.get("category","N/A"),
        "Total Assets":  f"${info.get('totalAssets',0)/1e9:.2f}B" if info.get('totalAssets') else "N/A",
        "Expense Ratio": f"{info.get('annualReportExpenseRatio',0)*100:.2f}%" if info.get('annualReportExpenseRatio') else "N/A",
        "52W High":      f"${info.get('fiftyTwoWeekHigh',0):.2f}" if info.get('fiftyTwoWeekHigh') else "N/A",
        "52W Low":       f"${info.get('fiftyTwoWeekLow',0):.2f}" if info.get('fiftyTwoWeekLow') else "N/A",
        "YTD Return":    f"{info.get('ytdReturn',0)*100:.2f}%" if info.get('ytdReturn') else "N/A",
        "3Y Return":     f"{info.get('threeYearAverageReturn',0)*100:.2f}%" if info.get('threeYearAverageReturn') else "N/A",
        "5Y Return":     f"{info.get('fiveYearAverageReturn',0)*100:.2f}%" if info.get('fiveYearAverageReturn') else "N/A",
    }

def get_news(ticker, company_name):
    """
    Pull all news headlines from the past 14 days.
    Headlines are used silently by Claude — not shown raw to the user.
    """
    from email.utils import parsedate_to_datetime
    headlines = []
    cutoff    = datetime.now() - timedelta(days=14)

    # Primary: Yahoo Finance RSS — grab everything, filter by date
    try:
        rss_url  = f"https://feeds.finance.yahoo.com/rss/2.0/headline?s={ticker}&region=US&lang=en-US"
        response = requests.get(rss_url, timeout=10, headers={"User-Agent": "Mozilla/5.0"})
        soup     = BeautifulSoup(response.content, "xml")
        for item in soup.find_all("item"):
            title    = item.find("title")
            pub_date = item.find("pubDate")
            if not title:
                continue
            try:
                article_date = parsedate_to_datetime(pub_date.text).replace(tzinfo=None)
                if article_date < cutoff:
                    continue
                date_str = article_date.strftime("%Y-%m-%d")
            except Exception:
                date_str = pub_date.text[:16] if pub_date else "N/A"
            headlines.append({"title": title.text.strip(), "date": date_str})
    except Exception:
        pass

    # Fallback: yfinance news — filter to past 14 days
    if not headlines:
        try:
            stock = yf.Ticker(ticker)
            for article in (stock.news or []):
                pub_ts       = article.get("providerPublishTime", 0)
                article_date = datetime.fromtimestamp(pub_ts)
                if article_date < cutoff:
                    continue
                headlines.append({
                    "title": article.get("title", "N/A"),
                    "date":  article_date.strftime("%Y-%m-%d")
                })
        except Exception:
            pass

    # Sort newest first
    headlines.sort(key=lambda x: x["date"], reverse=True)
    return headlines


def rank_news_by_impact(ticker, headlines):
    """
    Ask Claude to rank all headlines by market impact/sentiment significance.
    Returns top 10 most impactful, each with a one-line reason.
    Used for display — full list still goes to the main thesis.
    """
    if not headlines:
        return []

    # Build numbered list for Claude to rank
    numbered = "\n".join([f"{i+1}. [{h['date']}] {h['title']}" for i, h in enumerate(headlines)])

    prompt = f"""You are a financial analyst. Below are news headlines about {ticker} from the past 14 days.

{numbered}

Rank the TOP 10 most important headlines by their potential market impact on {ticker}'s stock price.
Consider: earnings, guidance, analyst upgrades/downgrades, macro events, regulatory news, product launches, legal issues.

Respond ONLY with a JSON array of exactly 10 objects (or fewer if less than 10 headlines exist), in this exact format:
[
  {{"rank": 1, "date": "YYYY-MM-DD", "title": "exact headline title", "reason": "one sentence on why this matters", "sentiment": "Bullish" or "Bearish" or "Neutral"}},
  ...
]
Return only valid JSON. No preamble, no explanation outside the JSON."""

    try:
        message = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=1500,
            messages=[{"role": "user", "content": prompt}]
        )
        import json, re
        raw  = message.content[0].text.strip()
        raw  = re.sub(r"```json|```", "", raw).strip()
        ranked = json.loads(raw)
        return ranked[:10]
    except Exception:
        # Fallback: return first 10 as-is without ranking
        return [{"rank": i+1, "date": h["date"], "title": h["title"],
                 "reason": "", "sentiment": "Neutral"}
                for i, h in enumerate(headlines[:10])]


# ════════════════════════════════════════════════════════════
# 📐 TECHNICAL INDICATORS
# ════════════════════════════════════════════════════════════

def calculate_indicators(df):
    # ── Moving Averages ───────────────────────────────────────
    df["MA20"]     = df["Close"].rolling(20).mean()
    df["MA50"]     = df["Close"].rolling(50).mean()
    df["MA200"]    = df["Close"].rolling(200).mean()

    # ── RSI ───────────────────────────────────────────────────
    delta          = df["Close"].diff()
    gain           = delta.where(delta > 0, 0).rolling(14).mean()
    loss           = -delta.where(delta < 0, 0).rolling(14).mean()
    df["RSI"]      = 100 - (100 / (1 + gain / loss))

    # ── MACD ──────────────────────────────────────────────────
    ema12          = df["Close"].ewm(span=12, adjust=False).mean()
    ema26          = df["Close"].ewm(span=26, adjust=False).mean()
    df["MACD"]     = ema12 - ema26
    df["Signal"]   = df["MACD"].ewm(span=9, adjust=False).mean()
    df["Hist"]     = df["MACD"] - df["Signal"]

    # ── Bollinger Bands ───────────────────────────────────────
    df["BB_Mid"]   = df["Close"].rolling(20).mean()
    bb_std         = df["Close"].rolling(20).std()
    df["BB_Upper"] = df["BB_Mid"] + 2 * bb_std
    df["BB_Lower"] = df["BB_Mid"] - 2 * bb_std

    # ── Volume MA ─────────────────────────────────────────────
    df["Vol_MA20"] = df["Volume"].rolling(20).mean()

    # ── Stochastic Oscillator (%K and %D) ─────────────────────
    low14          = df["Low"].rolling(14).min()
    high14         = df["High"].rolling(14).max()
    df["Stoch_K"]  = 100 * (df["Close"] - low14) / (high14 - low14 + 1e-9)
    df["Stoch_D"]  = df["Stoch_K"].rolling(3).mean()  # Signal line

    # ── VWAP (rolling 20-day) ─────────────────────────────────
    # True VWAP is intraday; we approximate a rolling daily VWAP
    typical_price  = (df["High"] + df["Low"] + df["Close"]) / 3
    df["VWAP"]     = (typical_price * df["Volume"]).rolling(20).sum() / df["Volume"].rolling(20).sum()

    return df


def get_support_resistance(df, lookback=252, n_levels=3):
    """
    Auto-detect support and resistance levels using local price pivots
    over the past year. Returns (support_levels, resistance_levels).
    """
    prices   = df["Close"].tail(lookback).values
    highs    = df["High"].tail(lookback).values
    lows     = df["Low"].tail(lookback).values
    current  = prices[-1]

    # Find pivot highs and lows using a simple rolling window
    resistance_candidates = []
    support_candidates    = []
    window = 10

    for i in range(window, len(prices) - window):
        if highs[i] == max(highs[i-window:i+window]):
            resistance_candidates.append(highs[i])
        if lows[i] == min(lows[i-window:i+window]):
            support_candidates.append(lows[i])

    # Filter: supports below current price, resistances above
    supports    = sorted([p for p in support_candidates    if p < current], reverse=True)[:n_levels]
    resistances = sorted([p for p in resistance_candidates if p > current])[:n_levels]

    # Cluster nearby levels (within 1.5%) to avoid clutter
    def cluster(levels, pct=0.015):
        clustered = []
        for lvl in levels:
            if not clustered or abs(lvl - clustered[-1]) / clustered[-1] > pct:
                clustered.append(lvl)
        return clustered

    return cluster(supports), cluster(resistances)


def get_fibonacci_levels(df, lookback=126):
    """
    Calculate Fibonacci retracement levels from the highest high
    and lowest low over the past ~6 months.
    """
    recent   = df.tail(lookback)
    high     = recent["High"].max()
    low      = recent["Low"].min()
    diff     = high - low
    ratios   = [0.0, 0.236, 0.382, 0.5, 0.618, 0.786, 1.0]
    levels   = {f"Fib {int(r*100)}%": round(high - diff * r, 2) for r in ratios}
    return levels, high, low


# ════════════════════════════════════════════════════════════
# 📈 CHART
# ════════════════════════════════════════════════════════════

def plot_charts(df, ticker):
    df_plot = df.tail(252).copy()

    # Pre-calculate support/resistance and fibonacci
    supports, resistances = get_support_resistance(df)
    fib_levels, fib_high, fib_low = get_fibonacci_levels(df)

    fig = plt.figure(figsize=(16, 16))
    fig.patch.set_facecolor("white")
    gs  = gridspec.GridSpec(4, 1, height_ratios=[3, 1, 1.2, 1], hspace=0.08)

    lc   = "#111111"
    gc   = "#e0e0e0"
    ab   = "#1a6fba"
    ag   = "#1a7a3a"
    ar   = "#cc2222"
    gold = "#b8860b"
    bb_c = "#9ecae1"
    vc   = "#7b2d8b"   # violet — VWAP
    fc   = "#e67e00"   # orange — Fibonacci

    # ── Panel 1: Price + MAs + BB + VWAP + Fib + S/R ─────────
    ax1 = fig.add_subplot(gs[0])
    ax1.set_facecolor("#fafafa")

    # Bollinger Bands
    ax1.fill_between(df_plot.index, df_plot["BB_Upper"], df_plot["BB_Lower"], alpha=0.10, color=bb_c)
    ax1.plot(df_plot.index, df_plot["BB_Upper"], color=bb_c, linewidth=0.8, alpha=0.8)
    ax1.plot(df_plot.index, df_plot["BB_Lower"], color=bb_c, linewidth=0.8, alpha=0.8)

    # Price and MAs
    ax1.plot(df_plot.index, df_plot["Close"], color="#222222", linewidth=1.6, label="Price", zorder=5)
    ax1.plot(df_plot.index, df_plot["MA20"],  color=gold, linewidth=1.0, label="MA20", linestyle="--")
    ax1.plot(df_plot.index, df_plot["MA50"],  color=ab,   linewidth=1.3, label="MA50")
    ax1.plot(df_plot.index, df_plot["MA200"], color=ar,   linewidth=1.3, label="MA200")

    # VWAP
    ax1.plot(df_plot.index, df_plot["VWAP"],  color=vc,   linewidth=1.2, label="VWAP (20D)", linestyle="-.")

    # Golden / Death Cross markers
    cross = df_plot[
        (df_plot["MA50"] > df_plot["MA200"]) !=
        (df_plot["MA50"].shift(1) > df_plot["MA200"].shift(1))
    ]
    for idx, row in cross.iterrows():
        color = ag if row["MA50"] > row["MA200"] else ar
        label = "☀ Golden" if row["MA50"] > row["MA200"] else "☠ Death"
        ax1.annotate(label, xy=(idx, row["Close"]), fontsize=7, color=color,
                     xytext=(10, 10), textcoords="offset points", fontweight="bold")

    # Fibonacci retracement lines
    fib_ratios_to_show = ["Fib 23%", "Fib 38%", "Fib 50%", "Fib 61%"]
    for key in fib_ratios_to_show:
        if key in fib_levels:
            ax1.axhline(fib_levels[key], color=fc, linewidth=0.7, linestyle=":", alpha=0.7)
            ax1.text(df_plot.index[-1], fib_levels[key], f" {key}", fontsize=6,
                     color=fc, va="center", alpha=0.9)

    # Support levels (dashed green)
    for lvl in supports:
        ax1.axhline(lvl, color=ag, linewidth=0.9, linestyle="--", alpha=0.5)
        ax1.text(df_plot.index[5], lvl, f" S ${lvl:.0f}", fontsize=6,
                 color=ag, va="bottom", alpha=0.8)

    # Resistance levels (dashed red)
    for lvl in resistances:
        ax1.axhline(lvl, color=ar, linewidth=0.9, linestyle="--", alpha=0.5)
        ax1.text(df_plot.index[5], lvl, f" R ${lvl:.0f}", fontsize=6,
                 color=ar, va="bottom", alpha=0.8)

    ax1.set_title(f"{ticker} — Technical Analysis Dashboard", color=lc, fontsize=14, pad=10, fontweight="bold")
    ax1.legend(loc="upper left", fontsize=7, facecolor="white", edgecolor=gc, ncol=2)
    ax1.tick_params(colors=lc, labelbottom=False)
    ax1.set_ylabel("Price ($)", color=lc)
    [s.set_edgecolor(gc) for s in ax1.spines.values()]
    ax1.grid(color=gc, linewidth=0.6)

    # ── Panel 2: Volume + Vol MA ──────────────────────────────
    ax2 = fig.add_subplot(gs[1], sharex=ax1)
    ax2.set_facecolor("#fafafa")
    colors = [ag if c >= o else ar for c, o in zip(df_plot["Close"], df_plot["Open"])]
    ax2.bar(df_plot.index, df_plot["Volume"], color=colors, alpha=0.6, width=1)
    ax2.plot(df_plot.index, df_plot["Vol_MA20"], color=gold, linewidth=1.2, label="Vol MA20")
    ax2.set_ylabel("Volume", color=lc, fontsize=8)
    ax2.tick_params(colors=lc, labelbottom=False)
    ax2.legend(loc="upper left", fontsize=7, facecolor="white", edgecolor=gc)
    [s.set_edgecolor(gc) for s in ax2.spines.values()]
    ax2.grid(color=gc, linewidth=0.6)

    # ── Panel 3: RSI + Stochastic ─────────────────────────────
    ax3 = fig.add_subplot(gs[2], sharex=ax1)
    ax3.set_facecolor("#fafafa")

    # RSI
    ax3.plot(df_plot.index, df_plot["RSI"], color=ab, linewidth=1.3, label="RSI (14)")
    ax3.axhline(70, color=ar, linewidth=0.9, linestyle="--", alpha=0.7)
    ax3.axhline(30, color=ag, linewidth=0.9, linestyle="--", alpha=0.7)
    ax3.fill_between(df_plot.index, df_plot["RSI"], 70, where=(df_plot["RSI"] >= 70), alpha=0.10, color=ar)
    ax3.fill_between(df_plot.index, df_plot["RSI"], 30, where=(df_plot["RSI"] <= 30), alpha=0.10, color=ag)

    # Stochastic on same panel (right y-axis twin)
    ax3b = ax3.twinx()
    ax3b.plot(df_plot.index, df_plot["Stoch_K"], color=vc,   linewidth=1.0, label="%K", alpha=0.8)
    ax3b.plot(df_plot.index, df_plot["Stoch_D"], color=gold, linewidth=1.0, label="%D", linestyle="--", alpha=0.8)
    ax3b.axhline(80, color=ar, linewidth=0.5, linestyle=":", alpha=0.5)
    ax3b.axhline(20, color=ag, linewidth=0.5, linestyle=":", alpha=0.5)
    ax3b.set_ylim(0, 100)
    ax3b.set_ylabel("Stoch", color=vc, fontsize=7)
    ax3b.tick_params(colors=vc, labelsize=7)

    ax3.set_ylim(0, 100)
    ax3.set_ylabel("RSI", color=lc, fontsize=8)
    ax3.tick_params(colors=lc, labelbottom=False)

    # Combined legend
    lines1, labels1 = ax3.get_legend_handles_labels()
    lines2, labels2 = ax3b.get_legend_handles_labels()
    ax3.legend(lines1 + lines2, labels1 + labels2,
               loc="upper left", fontsize=7, facecolor="white", edgecolor=gc)
    [s.set_edgecolor(gc) for s in ax3.spines.values()]
    ax3.grid(color=gc, linewidth=0.6)

    # ── Panel 4: MACD ─────────────────────────────────────────
    ax4 = fig.add_subplot(gs[3], sharex=ax1)
    ax4.set_facecolor("#fafafa")
    ax4.plot(df_plot.index, df_plot["MACD"],   color=ab,   linewidth=1.3, label="MACD")
    ax4.plot(df_plot.index, df_plot["Signal"], color=gold, linewidth=1.1, label="Signal")
    hc = [ag if v >= 0 else ar for v in df_plot["Hist"]]
    ax4.bar(df_plot.index, df_plot["Hist"], color=hc, alpha=0.5, width=1)
    ax4.axhline(0, color=lc, linewidth=0.5, linestyle=":")
    ax4.set_ylabel("MACD", color=lc, fontsize=8)
    ax4.tick_params(colors=lc)
    ax4.legend(loc="upper left", fontsize=7, facecolor="white", edgecolor=gc)
    [s.set_edgecolor(gc) for s in ax4.spines.values()]
    ax4.grid(color=gc, linewidth=0.6)

    plt.xticks(rotation=45, color=lc)
    plt.tight_layout()
    return fig


# ════════════════════════════════════════════════════════════
# 🤖 CLAUDE AI ANALYSIS
# ════════════════════════════════════════════════════════════

def analyze_with_claude(ticker, technicals, fundamentals, news_headlines, etf_flag, etf_info=None, fg=None):
    news_text = "\n".join([f"- [{h['date']}] {h['title']}" for h in news_headlines]) or "No recent news."
    if etf_flag and etf_info:
        fund_section = "ETF PROFILE:\n" + "\n".join([f"- {k}: {v}" for k,v in etf_info.items()])
    else:
        fund_section = "FUNDAMENTAL DATA:\n" + "\n".join([f"- {k}: {v}" for k,v in fundamentals.items()])

    # Build Fear & Greed section for prompt
    if fg and fg["score"] is not None:
        fg_section = f"""
MARKET SENTIMENT — CNN FEAR & GREED INDEX:
- Current Score:    {fg['score']} / 100
- Current Label:    {fg['label']}
- Last Week Score:  {fg['prev_week'] if fg['prev_week'] else 'N/A'}
- Last Month Score: {fg['prev_month'] if fg['prev_month'] else 'N/A'}
- Weekly Trend:     {fg['trend']}
- Interpretation:   {fg['mood']}"""
    else:
        fg_section = "\nMARKET SENTIMENT — CNN FEAR & GREED INDEX: Data unavailable."

    prompt = f"""
You are a senior portfolio manager and equity analyst with 20+ years of experience.
Analyze the following data for {ticker} ({'ETF' if etf_flag else 'Stock'}) and produce a detailed investment thesis.

TECHNICAL INDICATORS:
- Current Price:     ${technicals['price']:.2f}
- MA20:              ${technicals['ma20']:.2f}
- MA50:              ${technicals['ma50']:.2f}
- MA200:             ${technicals['ma200']:.2f}
- RSI (14-day):      {technicals['rsi']:.2f}
- Stochastic %K:     {technicals['stoch_k']:.2f}  ({'Overbought' if technicals['stoch_k'] > 80 else 'Oversold' if technicals['stoch_k'] < 20 else 'Neutral'})
- Stochastic %D:     {technicals['stoch_d']:.2f}
- MACD:              {technicals['macd']:.4f}
- MACD Signal:       {technicals['signal']:.4f}
- VWAP (20D):        ${technicals['vwap']:.2f}  ({'Price above VWAP — bullish' if technicals['price'] > technicals['vwap'] else 'Price below VWAP — bearish'})
- Bollinger Upper:   ${technicals['bb_upper']:.2f}
- Bollinger Lower:   ${technicals['bb_lower']:.2f}
- Price vs BB Mid:   {"Above" if technicals['price'] > technicals['bb_mid'] else "Below"}
- Volume vs 20D Avg: {"Above" if technicals['vol_ratio'] > 1 else "Below"} ({technicals['vol_ratio']:.1f}x)
- Support Levels:    {', '.join([f'${s:.2f}' for s in technicals['supports']]) or 'N/A'}
- Resistance Levels: {', '.join([f'${r:.2f}' for r in technicals['resistances']]) or 'N/A'}
- Fibonacci Range:   ${technicals['fib_low']:.2f} (low) — ${technicals['fib_high']:.2f} (high)
- Fib 23.6%:         ${technicals['fib_levels'].get('Fib 23%', 0):.2f}
- Fib 38.2%:         ${technicals['fib_levels'].get('Fib 38%', 0):.2f}
- Fib 50.0%:         ${technicals['fib_levels'].get('Fib 50%', 0):.2f}
- Fib 61.8%:         ${technicals['fib_levels'].get('Fib 61%', 0):.2f}
{fg_section}

{fund_section}

RECENT NEWS & CATALYSTS (past 14 days — {len(news_headlines)} articles):
{news_text}

Provide a structured analysis with these sections:

1. 📊 TECHNICAL SUMMARY
   - Trend (short/medium/long term)
   - Is price above or below VWAP? What does this signal for institutional momentum?
   - Are Stochastic %K and %D confirming or diverging from RSI?
   - Identify key support and resistance levels from the data — which are most critical?
   - Which Fibonacci level is price closest to? Is it acting as support or resistance?
   - Overall technical bias: Bullish / Neutral / Bearish

2. 😨 MARKET SENTIMENT & MACRO CONTEXT
   - Interpret the Fear & Greed score and its trend (rising fear vs rising greed)
   - Explain what this means for trade timing: e.g. extreme fear = contrarian buy signal, extreme greed = wait for pullback
   - Factor into whether NOW is a good time to enter, hold, or avoid regardless of the individual stock's setup
   - For ETFs: also assess macro tailwinds/headwinds for the ETF's sector or category

3. {"💼 FUNDAMENTAL ANALYSIS" if not etf_flag else "📋 ETF PROFILE ANALYSIS"}
   {"- Valuation vs sector, revenue growth, margins, balance sheet, FCF quality" if not etf_flag else "- Category exposure, performance vs benchmark, expense ratio value"}

4. 📰 NEWS SENTIMENT (past 14 days)
   - Summarize key themes and narrative across all headlines
   - Identify specific catalysts: earnings, guidance, analyst actions, regulatory news, macro events
   - Note sentiment trend over the 2-week window (improving, deteriorating, mixed)
   - Overall verdict: Bullish / Neutral / Bearish

5. ⚖️ BULL vs BEAR CASE
   Bull Case — 5 specific reasons to BUY, each grounded in the data above:
   1. [reason]
   2. [reason]
   3. [reason]
   4. [reason]
   5. [reason]

   Bear Case — 5 specific risks to be aware of, each grounded in the data above:
   1. [risk]
   2. [risk]
   3. [risk]
   4. [risk]
   5. [risk]

6. 🎯 INVESTMENT RECOMMENDATION
   - Verdict: STRONG BUY / BUY / HOLD / SELL / STRONG SELL
   - Conviction level: High / Medium / Low — and why
   - How does the current Fear & Greed reading affect your timing?
   - Ideal entry price range (be specific)
   - 3-month price target and 6-month price target
   - Stop-loss level with reasoning
   - Position sizing suggestion: Starter (1-2%), Core (3-5%), or Avoid
   - One sentence summary: what needs to happen for this thesis to play out

Be direct, specific, and data-driven. Write for an experienced investor.
"""
    message = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=3500,
        messages=[{"role": "user", "content": prompt}]
    )
    return message.content[0].text


# ════════════════════════════════════════════════════════════
# 🖥️ CORE FUNCTIONS CALLED BY GRADIO
# ════════════════════════════════════════════════════════════

def run_single_analysis(ticker_input):
    ticker = ticker_input.strip().upper()
    if not ticker:
        return None, "❌ Please enter a ticker symbol.", "", ""

    df, info, error = get_stock_data(ticker)
    if error:
        return None, f"❌ Error: {error}", "", ""

    etf_flag     = is_etf(info)
    company_name = info.get("longName", ticker)
    asset_label  = "ETF" if etf_flag else "Stock"

    # Fetch Fear & Greed alongside other data
    fg = get_fear_greed()

    df = calculate_indicators(df)
    latest = df.iloc[-1]

    # Support/Resistance and Fibonacci
    supports, resistances = get_support_resistance(df)
    fib_levels, fib_high, fib_low = get_fibonacci_levels(df)

    technicals = {
        "price":      latest["Close"],
        "ma20":       latest["MA20"],
        "ma50":       latest["MA50"],
        "ma200":      latest["MA200"],
        "rsi":        latest["RSI"],
        "macd":       latest["MACD"],
        "signal":     latest["Signal"],
        "bb_upper":   latest["BB_Upper"],
        "bb_lower":   latest["BB_Lower"],
        "bb_mid":     latest["BB_Mid"],
        "vol_ratio":  latest["Volume"] / latest["Vol_MA20"] if latest["Vol_MA20"] > 0 else 1,
        "stoch_k":    latest["Stoch_K"],
        "stoch_d":    latest["Stoch_D"],
        "vwap":       latest["VWAP"],
        "supports":   supports,
        "resistances":resistances,
        "fib_levels": fib_levels,
        "fib_high":   fib_high,
        "fib_low":    fib_low,
    }

    fundamentals = get_fundamentals(info) if not etf_flag else {}
    etf_info     = get_etf_info(info)     if etf_flag     else {}
    profile_data = etf_info if etf_flag else fundamentals

    rsi_label   = "🔴 Overbought" if technicals["rsi"] > 70 else "🟢 Oversold" if technicals["rsi"] < 30 else "⚪ Neutral"
    macd_label  = "📈 Bullish" if technicals["macd"] > technicals["signal"] else "📉 Bearish"
    ma_label    = "☀️ Golden Cross" if technicals["ma50"] > technicals["ma200"] else "☠️ Death Cross"
    stoch_label = "🔴 Overbought" if technicals["stoch_k"] > 80 else "🟢 Oversold" if technicals["stoch_k"] < 20 else "⚪ Neutral"
    vwap_label  = "📈 Above VWAP" if technicals["price"] > technicals["vwap"] else "📉 Below VWAP"

    sup_str = "  ".join([f"${s:.2f}" for s in technicals["supports"]])    or "N/A"
    res_str = "  ".join([f"${r:.2f}" for r in technicals["resistances"]]) or "N/A"
    fib_str = "\n".join([f"  {k:<12} ${v:.2f}" for k, v in list(technicals["fib_levels"].items())[1:-1]])  # skip 0% and 100%

    # Build Fear & Greed display block
    if fg["score"] is not None:
        fg_block = f"""
{'━'*45}
  {fg['emoji']} FEAR & GREED INDEX
{'━'*45}
  Current Score:   {fg['score']} / 100  ({fg['label']})
  Last Week:       {fg['prev_week'] if fg['prev_week'] else 'N/A'}  ({fg['trend']})
  Last Month:      {fg['prev_month'] if fg['prev_month'] else 'N/A'}
  Market Mood:     {fg['mood']}
  Data Source:     {fg.get('source', 'N/A')}"""
    else:
        fg_block = f"\n{'━'*45}\n  ❓ Fear & Greed Index unavailable — all sources failed\n{'━'*45}"

    metrics = f"""{'━'*45}
  {company_name} ({ticker}) — {asset_label}
  {datetime.now().strftime('%B %d, %Y  %I:%M %p')}
{'━'*45}

📊 TECHNICAL SNAPSHOT
  Current Price:   ${technicals['price']:.2f}
  MA20:            ${technicals['ma20']:.2f}
  MA50:            ${technicals['ma50']:.2f}
  MA200:           ${technicals['ma200']:.2f}
  MA Cross:        {ma_label}
  RSI:             {technicals['rsi']:.1f}  {rsi_label}
  Stochastic %K:   {technicals['stoch_k']:.1f}  {stoch_label}
  Stochastic %D:   {technicals['stoch_d']:.1f}
  MACD:            {technicals['macd']:.4f}  {macd_label}
  VWAP (20D):      ${technicals['vwap']:.2f}  {vwap_label}
  Volume:          {technicals['vol_ratio']:.1f}x 20-day average
  Bollinger Up:    ${technicals['bb_upper']:.2f}
  Bollinger Low:   ${technicals['bb_lower']:.2f}

{'━'*45}
  📐 SUPPORT & RESISTANCE
{'━'*45}
  Support:         {sup_str}
  Resistance:      {res_str}

{'━'*45}
  🌀 FIBONACCI LEVELS (6M range: ${technicals['fib_low']:.2f} — ${technicals['fib_high']:.2f})
{'━'*45}
{fib_str}
{fg_block}

{'━'*45}
  {"📋 ETF PROFILE" if etf_flag else "💼 FUNDAMENTALS"}
{'━'*45}
""" + "\n".join([f"  {k:<20} {v}" for k, v in profile_data.items()])

    news = get_news(ticker, company_name)

    # Rank news by market impact (uses a separate Claude call)
    ranked_news = rank_news_by_impact(ticker, news)

    # Build display for UI — top 10 by impact with sentiment tag
    sentiment_icon = {"Bullish": "🟢", "Bearish": "🔴", "Neutral": "⚪"}
    if ranked_news:
        news_lines = [f"Top {len(ranked_news)} headlines by market impact (past 14 days — {len(news)} total scraped):\n"]
        for item in ranked_news:
            icon   = sentiment_icon.get(item.get("sentiment","Neutral"), "⚪")
            reason = f"  → {item['reason']}" if item.get("reason") else ""
            news_lines.append(f"{icon} [{item['date']}] {item['title']}{reason}\n")
        news_ui = "\n".join(news_lines)
    else:
        news_ui = "⚠️ No news found for this ticker in the past 14 days."

    fig = plot_charts(df, ticker)

    # Full unranked list goes to Claude for the main thesis
    ai_analysis = analyze_with_claude(ticker, technicals, fundamentals, news, etf_flag, etf_info or None, fg)
    disclaimer  = "\n\n⚠️ DISCLAIMER: AI-generated analysis for educational purposes only. Not financial advice."
    return fig, metrics, ai_analysis + disclaimer, news_ui


def run_watchlist_analysis(watchlist_input):
    tickers = [t.strip().upper() for t in watchlist_input.split(",") if t.strip()]
    if not tickers:
        return "❌ Please enter at least one ticker."

    rows = [
        f"{'Ticker':<8} {'Type':<6} {'Price':<10} {'RSI':<7} {'MA Cross':<16} {'MACD':<12} {'Score':<7} Verdict",
        "─" * 78
    ]
    for ticker in tickers:
        try:
            df, info, error = get_stock_data(ticker)
            if error or df is None:
                rows.append(f"{ticker:<8} ❌ Could not fetch")
                continue
            df = calculate_indicators(df)
            l  = df.iloc[-1]
            score = sum([
                1 if l["MA50"] > l["MA200"] else 0,
                1 if 40 < l["RSI"] < 65 else 0,
                1 if l["MACD"] > l["Signal"] else 0,
            ])
            verdict  = "🟢 BUY"  if score == 3 else "🟡 HOLD" if score == 2 else "🔴 AVOID"
            ma_cross = "Golden Cross" if l["MA50"] > l["MA200"] else "Death Cross"
            macd_dir = "Bullish"     if l["MACD"] > l["Signal"] else "Bearish"
            asset    = "ETF"         if is_etf(info) else "Stock"
            rows.append(f"{ticker:<8} {asset:<6} ${l['Close']:<9.2f} {l['RSI']:<7.1f} {ma_cross:<16} {macd_dir:<12} {score}/3    {verdict}")
        except Exception as e:
            rows.append(f"{ticker:<8} ❌ Error: {e}")

    rows.append("")
    rows.append(f"💰 Cost to deep-dive all: ~${len(tickers)*0.01:.2f}  |  Screener cost: FREE")
    rows.append("👉 Switch to 'Single Analysis' tab to deep-dive your top picks.")
    return "\n".join(rows)


In [3]:

# ════════════════════════════════════════════════════════════
# 🖥️ GRADIO UI
# ════════════════════════════════════════════════════════════

LIGHT_CSS = """
body, .gradio-container { background-color: #ffffff !important; color: #111111 !important; font-family: 'Segoe UI', sans-serif; }
button.primary { background: #1a6fba !important; border: none !important; color: white !important; font-weight: bold; border-radius: 6px !important; }
h1, h2, h3 { color: #1a3a5c !important; }
label { color: #333333 !important; font-weight: 600; }
.gr-box, .gr-form, .gr-panel { background: #f8f9fa !important; border: 1px solid #dee2e6 !important; border-radius: 8px !important; }
textarea, input[type=text] { background: #ffffff !important; color: #111111 !important; border: 1px solid #cccccc !important; border-radius: 6px !important; }
"""

with gr.Blocks(css=LIGHT_CSS, theme=gr.themes.Default(), title="📈 AI Stock Analyzer") as app:

    gr.Markdown("""
    # 📈 AI Stock & ETF Analyzer
    **Powered by Claude AI · Yahoo Finance · Real-time Data**
    """)

    with gr.Tabs():

        # ── Tab 1: Single Ticker ──────────────────────────────
        with gr.Tab("🔍 Single Analysis"):
            with gr.Row():
                ticker_input = gr.Textbox(
                    label="Ticker Symbol",
                    placeholder="e.g. AAPL, NVDA, SPYM, SPY",
                    scale=4
                )
                analyze_btn = gr.Button("🚀 Analyze", variant="primary", scale=1)

            with gr.Row():
                with gr.Column(scale=1):
                    metrics_out = gr.Textbox(
                        label="📊 Technical & Fundamental Snapshot",
                        lines=25,
                        interactive=False
                    )
                    news_out = gr.Textbox(
                        label="📰 Recent News Headlines",
                        lines=10,
                        interactive=False
                    )
                with gr.Column(scale=2):
                    chart_out = gr.Plot(label="📈 Technical Chart")
                    ai_out    = gr.Textbox(
                        label="🤖 Claude AI Investment Thesis",
                        lines=28,
                        interactive=False
                    )

            analyze_btn.click(
                fn=run_single_analysis,
                inputs=[ticker_input],
                outputs=[chart_out, metrics_out, ai_out, news_out]
            )
            ticker_input.submit(
                fn=run_single_analysis,
                inputs=[ticker_input],
                outputs=[chart_out, metrics_out, ai_out, news_out]
            )

        # ── Tab 2: Watchlist Screener ─────────────────────────
        with gr.Tab("📋 Watchlist Screener"):
            gr.Markdown("Enter multiple tickers to get a free quick scorecard. Then switch to **Single Analysis** to deep-dive your top picks.")
            with gr.Row():
                watchlist_input = gr.Textbox(
                    label="Your Watchlist",
                    placeholder="e.g. AAPL, NVDA, MSFT, SPY, SPYM, TSLA",
                    scale=4
                )
                watchlist_btn = gr.Button("📋 Screen All", variant="primary", scale=1)

            watchlist_out = gr.Textbox(
                label="📊 Watchlist Scorecard",
                lines=20,
                interactive=False
            )
            watchlist_btn.click(fn=run_watchlist_analysis, inputs=[watchlist_input], outputs=[watchlist_out])
            watchlist_input.submit(fn=run_watchlist_analysis, inputs=[watchlist_input], outputs=[watchlist_out])

        # ── Tab 3: How to Use ─────────────────────────────────
        with gr.Tab("ℹ️ How to Use"):
            gr.Markdown("""
            ## How to Use This Tool

            ### 🔍 Single Analysis
            1. Type any stock or ETF ticker (e.g. `AAPL`, `NVDA`, `SPY`, `SPYM`)
            2. Press **Enter** or click **Analyze**
            3. Wait ~15–20 seconds for the full report
            4. Review the chart, snapshot, news, and Claude's recommendation

            ### 📋 Watchlist Screener
            1. Enter multiple tickers separated by commas
            2. Click **Screen All** — completely free, no Claude API used
            3. Review the scorecard ranked by signal strength (score out of 3)
            4. Switch to **Single Analysis** to deep-dive your best picks

            ---
            ### 📊 Signal Guide

            | Signal | Meaning |
            |---|---|
            | ☀️ Golden Cross | MA50 > MA200 — bullish long-term trend |
            | ☠️ Death Cross | MA50 < MA200 — bearish long-term trend |
            | RSI < 30 | Oversold — potential bounce |
            | RSI > 70 | Overbought — potential pullback |
            | MACD Bullish | Short-term momentum positive |
            | Bollinger Bands | Price near upper = stretched, near lower = oversold |
            | Stochastic > 80 | Overbought — confirms RSI or signals divergence |
            | Stochastic < 20 | Oversold — potential reversal signal |
            | Price above VWAP | Institutional buying pressure — bullish |
            | Price below VWAP | Institutional selling pressure — bearish |
            | Fibonacci 61.8% | Strongest retracement support/resistance level |

            ---
            ### 💰 Estimated Cost Per Run

            | Action | Claude Calls | Est. Cost |
            |---|---|---|
            | Watchlist Screener | 0 | **FREE** |
            | News ranking (2-week headlines) | 1 | ~$0.011 |
            | Main investment thesis | 1 | ~$0.033 |
            | **Full single analysis** | **2** | **~$0.04** |

            **Real world usage examples:**
            - 1 analysis = ~$0.04
            - 5 tickers/morning = ~$0.22/day
            - 5 tickers/day for a month = ~$6.60/month
            - $10 in API credits ≈ ~250 full analyses

            **💡 Tip:** Use the Watchlist Screener first (free) to identify your top 2-3 picks, then only deep-dive those. At 2 analyses/day that's ~$2.50/month.

            ---
            ⚠️ *For educational purposes only. Not financial advice. Always do your own research.*
            """)

# ── Launch ────────────────────────────────────────────────────
print("\n🚀 Launching AI Stock Analyzer...")
if IN_COLAB:
    print("Your app window will appear below in a few seconds.\n")
    app.launch(share=False, debug=False)
else:
    print("Local server starting — click the URL below to open the app.\n")
    app.launch(share=False, debug=False, inbrowser=True)



🚀 Launching AI Stock Analyzer...
Local server starting — click the URL below to open the app.

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.
